# Импорты и настройка среды

In [1]:
import random
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
import faiss

from sklearn.metrics import pairwise_distances
import matplotlib.pyplot as plt

# seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# device
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

# База знаний (Wikipedia)

Используется небольшая тематическая база знаний на основе статей Wikipedia по теме: Artificial Intelligence / Machine Learning / NLP
Это хорошая база для retrieval, потому что: тексты содержательные; есть чёткие факты; можно формулировать разнообразные вопросы.

In [2]:
documents = [
    """Artificial intelligence (AI) is intelligence demonstrated by machines, 
    in contrast to natural intelligence displayed by humans and animals. 
    AI includes learning, reasoning, and self-correction.""",
    
    """Machine learning is a field of artificial intelligence that uses statistical techniques 
    to give computer systems the ability to learn from data without being explicitly programmed.""",
    
    """Natural language processing (NLP) is a subfield of linguistics, computer science, 
    and artificial intelligence concerned with the interactions between computers and human language.""",
    
    """Deep learning is part of a broader family of machine learning methods based on artificial neural networks 
    with representation learning.""",
    
    """Neural networks are computing systems inspired by biological neural networks 
    that constitute animal brains.""",
    
    """Supervised learning is the machine learning task of learning a function 
    that maps input to output based on example input-output pairs.""",
    
    """Unsupervised learning is a type of algorithm used to draw inferences from datasets 
    consisting of input data without labeled responses.""",
    
    """Reinforcement learning is an area of machine learning concerned with how agents 
    take actions in an environment to maximize reward.""",
    
    """Transformers are a type of neural network architecture used in NLP 
    that rely on self-attention mechanisms.""",
    
    """Large language models (LLMs) are deep learning models trained on vast amounts of text 
    to understand and generate human language."""
]

print(f"Number of documents: {len(documents)}")
print("\nExample document:\n", documents[0][:200])

Number of documents: 10

Example document:
 Artificial intelligence (AI) is intelligence demonstrated by machines, 
    in contrast to natural intelligence displayed by humans and animals. 
    AI includes learning, reasoning, and self-correcti


# Чанкинг документов

Документы разбиваются на чанки фиксированного размера.
Параметры: chunk_size = 200, overlap = 50

In [3]:
def chunk_text(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    
    return chunks

chunks = []
for doc_id, doc in enumerate(documents):
    for chunk in chunk_text(doc):
        chunks.append({
            "doc_id": doc_id,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunks_df))
chunks_df.head()

print("Example chunks for first document:")
for chunk in chunk_text(documents[0]):
    print("-", chunk)

Total chunks: 13
Example chunks for first document:
- Artificial intelligence (AI) is intelligence demonstrated by machines, 
    in contrast to natural intelligence displayed by humans and animals. 
    AI includes learning, reasoning, and self-correcti
- AI includes learning, reasoning, and self-correction.


Чанкинг позволяет разбить документы на более мелкие фрагменты, 
что улучшает качество retrieval за счёт более точного поиска.

# Эмбеддинги + FAISS

Используется модель:
`sentence-transformers/all-MiniLM-L6-v2`

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

embeddings = model.encode(
    chunks_df["text"].tolist(),
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index size: 13


# Retrieval функция

In [5]:
def search(query, top_k=3):
    query_vec = model.encode([query]).astype("float32")
    
    distances, indices = index.search(query_vec, top_k)
    
    results = chunks_df.iloc[indices[0]]
    return results

# Примеры retrieval

In [6]:
queries = [
    "What is machine learning?",
    "What are transformers?",
    "What is reinforcement learning?"
]

for q in queries:
    print("\nQUERY:", q)
    print(search(q)["text"].values)


QUERY: What is machine learning?
['Machine learning is a field of artificial intelligence that uses statistical techniques \n    to give computer systems the ability to learn from data without being explicitly programmed.'
 'Supervised learning is the machine learning task of learning a function \n    that maps input to output based on example input-output pairs.'
 'Reinforcement learning is an area of machine learning concerned with how agents \n    take actions in an environment to maximize reward.']

QUERY: What are transformers?
['Transformers are a type of neural network architecture used in NLP \n    that rely on self-attention mechanisms.'
 'ween computers and human language.'
 'Deep learning is part of a broader family of machine learning methods based on artificial neural networks \n    with representation learning.']

QUERY: What is reinforcement learning?
['Reinforcement learning is an area of machine learning concerned with how agents \n    take actions in an environment t

# Контрольные запросы + метрики

In [7]:
eval_queries = [
    {"query": "What is NLP?", "doc_id": 2},
    {"query": "Define machine learning", "doc_id": 1},
    {"query": "What are neural networks?", "doc_id": 4},
]

top_k = 3
results = []

for item in eval_queries:
    res = search(item["query"], top_k)
    
    retrieved_doc_ids = res["doc_id"].tolist()
    
    hit = int(item["doc_id"] in retrieved_doc_ids)

    recall = hit  
    
    results.append({
        "query": item["query"],
        "expected_doc": item["doc_id"],
        "retrieved": retrieved_doc_ids,
        "hit@k": hit,
        "recall@k": recall
    })

eval_df = pd.DataFrame(results)
eval_df

,query,expected_doc,retrieved,hit@k,recall@k
0,What is NLP?,2,"[2, 8, 9]",1,1
1,Define machine learning,1,"[1, 5, 0]",1,1
2,What are neural networks?,4,"[4, 3, 8]",1,1


# Эксперимент с параметрами

Сравниваются два значения chunk_size: 200, 300

Оценивается hit@k

In [8]:
def build_pipeline(chunk_size):
    chunks = []
    
    for doc_id, doc in enumerate(documents):
        for chunk in chunk_text(doc, chunk_size=chunk_size):
            chunks.append({
                "doc_id": doc_id,
                "text": chunk
            })
    
    df = pd.DataFrame(chunks)
    
    emb = model.encode(df["text"].tolist())
    emb = np.array(emb).astype("float32")
    
    idx = faiss.IndexFlatL2(emb.shape[1])
    idx.add(emb)
    
    return df, idx


def evaluate(df, idx):
    hits = []
    
    for item in eval_queries:
        q_vec = model.encode([item["query"]]).astype("float32")
        _, indices = idx.search(q_vec, 3)
        
        retrieved = df.iloc[indices[0]]["doc_id"].tolist()
        hits.append(int(item["doc_id"] in retrieved))
    
    return np.mean(hits)


df_200, idx_200 = build_pipeline(200)
df_300, idx_300 = build_pipeline(300)

score_200 = evaluate(df_200, idx_200)
score_300 = evaluate(df_300, idx_300)

print("chunk_size=200:", score_200)
print("chunk_size=300:", score_300)

chunk_size=200: 1.0
chunk_size=300: 1.0


# Обновление базы знаний и переиндексация

In [9]:
queries_test = [q["query"] for q in eval_queries]

def get_results(df, idx):
    results = {}
    for q in queries_test:
        q_vec = model.encode([q]).astype("float32")
        _, indices = idx.search(q_vec, 3)
        results[q] = df.iloc[indices[0]]["doc_id"].tolist()
    return results


# ДО обновления
before_results = get_results(chunks_df, index)

# обновление базы
new_docs = [
    "BERT is a transformer-based model designed for NLP understanding tasks.",
    "GPT is a generative language model used for text generation."
]

documents_updated = documents + new_docs

# rebuild
chunks_updated = []
for doc_id, doc in enumerate(documents_updated):
    for chunk in chunk_text(doc):
        chunks_updated.append({"doc_id": doc_id, "text": chunk})

df_updated = pd.DataFrame(chunks_updated)

emb_updated = model.encode(df_updated["text"].tolist())
emb_updated = np.array(emb_updated).astype("float32")

index_updated = faiss.IndexFlatL2(emb_updated.shape[1])
index_updated.add(emb_updated)

# ПОСЛЕ обновления
after_results = get_results(df_updated, index_updated)

comparison = []
for q in queries_test:
    comparison.append({
        "query": q,
        "before_retrieved_sources": before_results[q],
        "after_retrieved_sources": after_results[q],
        "changed": before_results[q] != after_results[q]
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

,query,before_retrieved_sources,after_retrieved_sources,changed
0,What is NLP?,"[2, 8, 9]","[2, 10, 8]",True
1,Define machine learning,"[1, 5, 0]","[1, 5, 0]",False
2,What are neural networks?,"[4, 3, 8]","[4, 3, 8]",False


Добавление новых документов расширяет базу знаний 
и может улучшить retrieval для ранее не покрытых запросов.

# Mini-RAG

In [10]:
def mini_rag(query, top_k=3):
    q_vec = model.encode([query]).astype("float32")
    _, indices = index_updated.search(q_vec, top_k)
    
    retrieved = df_updated.iloc[indices[0]]
    context = " ".join(retrieved["text"].tolist())
    
    answer = context[:300]
    
    return {
        "question": query,
        "answer": answer,
        "sources": retrieved["doc_id"].tolist()
    }


rag_examples = [
    mini_rag("What is BERT?"),
    mini_rag("What is machine learning?"),
    mini_rag("What are transformers?")
]

rag_df = pd.DataFrame(rag_examples)
rag_df

,question,answer,sources
0,What is BERT?,BERT is a transformer-based model designed for...,"[10, 2, 8]"
1,What is machine learning?,Machine learning is a field of artificial inte...,"[1, 5, 7]"
2,What are transformers?,Transformers are a type of neural network arch...,"[8, 10, 2]"


Качество ответа напрямую зависит от качества retrieval. 
Если извлечённые чанки нерелевантны, итоговый ответ будет некорректным.

# Сохранение артефактов

In [11]:
import os
os.makedirs("artifacts", exist_ok=True)

# retrieval_eval
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)

# rag examples
rag_df.to_csv("artifacts/rag_examples.csv", index=False)

# before/after
comparison_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)